# ecg2mcu quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WalidGabtni/ecg2mcu/blob/main/notebooks/ecg2mcu_quickstart.ipynb)

Train an ECG classifier, quantize it to INT8 for a microcontroller, and **verify that the quantized model still works**. This notebook runs the whole pipeline on a small bundled sample in about two minutes, then shows the verification catching two broken exports.

The sample is only 550 ECG windows, so the accuracy numbers here are not meaningful. They show that the pipeline works. To train on the full PTB-XL dataset, see the [README](https://github.com/WalidGabtni/ecg2mcu#train-on-the-full-ptb-xl-dataset).

## 1. Install

In [1]:
%pip install -q "ecg2mcu[export] @ git+https://github.com/WalidGabtni/ecg2mcu.git"

## 2. Load the sample and the configuration

In [2]:
import os
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from IPython.display import Markdown, display
from ecg2mcu.config import load_config
from ecg2mcu.data import load_splits
from ecg2mcu.train import train
from ecg2mcu.export import export_model
from ecg2mcu.verify import verify

import contextlib, io
def quiet_export(*args, **kwargs):
    """export_model without TensorFlow's very chatty converter output."""
    with contextlib.redirect_stdout(io.StringIO()):
        return export_model(*args, **kwargs)

cfg = load_config(overrides={
    "train.epochs": 15,
    "export.out_dir": "runs/demo",
    "export.tolerance_auc": 0.05,   # loose: the sample test set is tiny
})
splits = load_splits(cfg)
print(f"train={len(splits.x_train)}  val={len(splits.x_val)}  test={len(splits.x_test)}  "
      f"window shape={splits.x_train.shape[1:]}  (1 lead, 10 s at 100 Hz)")

train=300  val=100  test=150  window shape=(1, 1000)  (1 lead, 10 s at 100 Hz)


## 3. Train the 1D-CNN (float32)

In [3]:
model, summary = train(cfg, splits, cfg.export.out_dir)

[train] device=cuda channels=1 params=201,381 train=300 val=100 test=150
[train] epoch   1/15  train_loss=0.6601  val_loss=0.6769
[train] epoch   2/15  train_loss=0.6119  val_loss=0.6502
[train] epoch   3/15  train_loss=0.6021  val_loss=0.6356
[train] epoch   4/15  train_loss=0.5736  val_loss=0.6107
[train] epoch   5/15  train_loss=0.5538  val_loss=0.5746
[train] epoch   6/15  train_loss=0.5300  val_loss=0.5418
[train] epoch   7/15  train_loss=0.5169  val_loss=0.5217
[train] epoch   8/15  train_loss=0.5031  val_loss=0.5278
[train] epoch   9/15  train_loss=0.5000  val_loss=0.5227
[train] epoch  10/15  train_loss=0.4850  val_loss=0.5214
[train] epoch  11/15  train_loss=0.4832  val_loss=0.5874
[train] epoch  12/15  train_loss=0.4653  val_loss=0.5587
[train] epoch  13/15  train_loss=0.4602  val_loss=0.5224
[train] epoch  14/15  train_loss=0.4517  val_loss=0.5784
[train] epoch  15/15  train_loss=0.4443  val_loss=0.5857
[train] float32 test macro AUC = 0.7573  -> runs\demo\model.pth


## 4. Export to INT8 TensorFlow Lite

The model is converted with post-training quantization. The converter needs a **calibration set**: real input windows used to choose the INT8 scales. A good export uses representative windows.

In [4]:
calibration = splits.x_train[:100]
info = quiet_export(model, calibration, cfg.export.out_dir)
print(f"model.tflite: {info['model_bytes'] / 1024:.1f} KB")

model.tflite: 226.6 KB


## 5. Verify

Run the INT8 model and the float model on the held-out test windows and compare. The report also estimates the flash and RAM the model needs on the device.

In [5]:
report = verify(model, f"{cfg.export.out_dir}/model.tflite", splits.x_test, splits.y_test, cfg, cfg.export.out_dir)
display(Markdown(open(f"{cfg.export.out_dir}/verify_report.md").read()))

# ecg2mcu verification: PASSED

Test windows: 150

| Metric | Value |
|---|---|
| Float32 macro AUC | 0.7573 |
| INT8 macro AUC | 0.7583 |
| AUC drop | -0.0010 |
| Mean / max probability difference | 0.0028 / 0.0324 |
| Float logit range | [-4.6, 2.4] |
| INT8 output range | [-5.1, 2.7] |
| Flash (model) | 226.6 KB |
| Tensor arena (lower bound / recommended) | 46.9 KB / 72 KB |

The arena figures are estimates; confirm on the device (`AllocateTensors()` must succeed).


## 6. Break it on purpose

A quantized model can fail without any error. This is what happened to the thesis prototype this project grew out of: the INT8 model scored chance level on a real test set, and nothing complained.

**Case A: calibration data in the wrong scale.** Here the calibration windows are 50 times too small, as if they came from a different preprocessing step.

In [6]:
bad = "runs/bad_scale"
quiet_export(model, splits.x_train[:100] * 0.02, bad)
verify(model, f"{bad}/model.tflite", splits.x_test, splits.y_test, cfg, bad)
display(Markdown(open(f"{bad}/verify_report.md").read()))

# ecg2mcu verification: FAILED

Test windows: 150

| Metric | Value |
|---|---|
| Float32 macro AUC | 0.7573 |
| INT8 macro AUC | 0.5964 |
| AUC drop | +0.1609 |
| Mean / max probability difference | 0.2899 / 0.7377 |
| Float logit range | [-4.6, 2.4] |
| INT8 output range | [-3.7, 1.1] |
| Flash (model) | 226.9 KB |
| Tensor arena (lower bound / recommended) | 46.9 KB / 72 KB |

## Problems
- INT8 macro AUC dropped by 0.1609 (limit 0.05)
- mean probability difference 0.2899 exceeds 0.05
- 5% of float logits fall outside the INT8 output range [-3.74, 1.10]

The arena figures are estimates; confirm on the device (`AllocateTensors()` must succeed).


**Case B: an unrepresentative calibration set.** The same single window repeated 100 times. The INT8 output range collapses, so most predictions are clipped. This is the failure that went unnoticed on the real device.

In [7]:
bad = "runs/bad_calibration"
quiet_export(model, np.repeat(splits.x_train[:1], 100, axis=0), bad)
verify(model, f"{bad}/model.tflite", splits.x_test, splits.y_test, cfg, bad)
display(Markdown(open(f"{bad}/verify_report.md").read()))

# ecg2mcu verification: FAILED

Test windows: 150

| Metric | Value |
|---|---|
| Float32 macro AUC | 0.7573 |
| INT8 macro AUC | 0.7473 |
| AUC drop | +0.0101 |
| Mean / max probability difference | 0.0744 / 0.4629 |
| Float logit range | [-4.6, 2.4] |
| INT8 output range | [-1.6, 0.0] |
| Flash (model) | 226.9 KB |
| Tensor arena (lower bound / recommended) | 46.9 KB / 72 KB |

## Problems
- mean probability difference 0.0744 exceeds 0.05
- 56% of float logits fall outside the INT8 output range [-1.59, 0.00]

The arena figures are estimates; confirm on the device (`AllocateTensors()` must succeed).


In a script or CI pipeline, `ecg2mcu verify` exits with a non-zero status when a check fails, so a bad export cannot reach the firmware unnoticed.

## Next steps

- Train on the full PTB-XL dataset (Lead I, six leads or twelve): `python scripts/download_ptbxl.py`, then `ecg2mcu run --config configs/ptbxl_lead1.toml`.
- Copy the generated `cnn1d_model.h` into the ESP32 firmware template in `firmware/ecg_inference/`.
- How the checks work: [docs/how-it-works.md](https://github.com/WalidGabtni/ecg2mcu/blob/main/docs/how-it-works.md).

Research code, not a medical device.